# ARC SFT(LoRA) Notebook


## 1. Config and Environment
Set hyperparameters and random seeds for reproducibility.


In [ ]:
# Adjust hyperparameters and paths here
from dataclasses import dataclass
import os
import random
import torch

@dataclass
class Config:
    # data
    data_root: str = 'data'
    train_split: str = 'training'
    eval_split: str = 'evaluation'
    max_train_tasks: int = 0
    max_eval_tasks: int = 0
    use_augmentation: bool = False
    aug_samples_per_task: int = 1
    aug_seed: int = 0

    # model
    model_id: str = ''
    max_seq_len: int = 1024
    max_new_tokens: int = 512

    # LoRA
    lora_r: int = 64
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: tuple = (
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    )

    # training
    output_dir: str = 'outputs/arc_lora_sft'
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 1e-4
    num_train_epochs: int = 3
    logging_steps: int = 10
    save_steps: int = 200

cfg = Config()

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

print(os.getcwd())

## 2. Data Preparation
Build SFT samples (prompt/target); optional augmentation.


In [ ]:
# Build SFT training samples
from eval.core import GridCodec
from eval.core import ARCDataset
from eval.data_aug import AugmentedARCDataset
from ARChitects.sft_utils import build_sft_samples

codec = GridCodec()

base_train = ARCDataset(root=cfg.data_root, split=cfg.train_split, max_tasks=cfg.max_train_tasks)
if cfg.use_augmentation:
    train_source = AugmentedARCDataset(
        base=base_train,
        samples_per_task=cfg.aug_samples_per_task,
        seed=cfg.aug_seed,
    )
else:
    train_source = base_train

samples = build_sft_samples(train_source, codec)
print('samples:', len(samples))


## 3. Load Model and Tokenizer


In [ ]:
# Load model and tokenizer; ensure special tokens
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(cfg.model_id, use_fast=False)

# ensure special tokens for the prompt format
specials = {'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}
tokenizer.add_special_tokens(specials)

if tokenizer.eos_token_id is None:
    eos_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
    if eos_id is not None and eos_id != tokenizer.unk_token_id:
        tokenizer.eos_token_id = eos_id

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token else '<|im_end|>'

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_id,
    dtype=getattr(torch, 'bfloat16', None),
    device_map='auto',
)

if len(tokenizer) > model.get_input_embeddings().num_embeddings:
    model.resize_token_embeddings(len(tokenizer))

model.config.use_cache = False


## Baseline

In [ ]:
# Baseline evaluation (before LoRA)
from eval.solvers import RawSolver
from eval.models import HuggingFaceBackendTextGenerator
from eval.core import run_evaluation

def get_truth(task, context):
    tests = task.get('test') or []
    if len(tests) != 1:
        return None
    return tests[0].get('output')

def get_task_id(task, context):
    return task.get('task_id', context.get('index'))

eval_dataset = ARCDataset(root=cfg.data_root, split=cfg.eval_split, max_tasks=cfg.max_eval_tasks)
baseline_model = HuggingFaceBackendTextGenerator(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=cfg.max_new_tokens,
)
baseline_solver = RawSolver(model=baseline_model, codec=codec)
baseline_reports = run_evaluation(
    dataset=eval_dataset,
    solver=baseline_solver,
    get_truth=get_truth,
    get_task_id=get_task_id,
    output_dir=os.path.join(cfg.output_dir, 'reports_baseline'),
    model_id=cfg.model_id,
    model_key='baseline',
    viz_failures=True,
)

print(baseline_reports['summary'])

## 4. Inject LoRA


In [ ]:
# Freeze base; train LoRA only
from peft import LoraConfig, get_peft_model

for p in model.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=list(cfg.lora_target_modules),
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 5. Dataset and Collator
Build labels with prompt masking.


In [ ]:
# Build PyTorch Dataset and collator
from ARChitects.sft_utils import ArcSFTDataset, collate_sft

train_dataset = ArcSFTDataset(samples, tokenizer, max_seq_len=cfg.max_seq_len)
def collate_fn(batch):
    return collate_sft(batch, tokenizer=tokenizer)


## 6. Training


In [ ]:
# Train LoRA
from transformers import TrainingArguments, Trainer

bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_train_epochs,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    bf16=bf16_ok,
    fp16=torch.cuda.is_available() and not bf16_ok,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
)

trainer.train()


## 7. Save
Save LoRA adapter and tokenizer.


In [ ]:
# Train LoRA adapter
adapter_dir = os.path.join(cfg.output_dir, 'lora_adapter')
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('saved:', adapter_dir)


## 8. Evaluation


In [ ]:
# Evaluate with eval/ and write reports
from eval.solvers import RawSolver
from eval.models import HuggingFaceBackendTextGenerator
from eval.core import run_evaluation

def get_truth(task, context):
    tests = task.get('test') or []
    if len(tests) != 1:
        return None
    return tests[0].get('output')

def get_task_id(task, context):
    return task.get('task_id', context.get('index'))

eval_dataset = ARCDataset(root=cfg.data_root, split=cfg.eval_split, max_tasks=cfg.max_eval_tasks)
eval_model = HuggingFaceBackendTextGenerator(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=cfg.max_new_tokens,
)
solver = RawSolver(model=eval_model, codec=codec)

reports = run_evaluation(
    dataset=eval_dataset,
    solver=solver,
    get_truth=get_truth,
    get_task_id=get_task_id,
    output_dir=os.path.join(cfg.output_dir, 'reports'),
    model_id=cfg.model_id,
    model_key='lora_sft',
    viz_failures=True,
)

print(reports['summary'])